In [1]:
import os
import sys
import time

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED

from topnum.regularizers import (
    FastFixPhiRegularizer, DecorrelateWithOtherPhiRegularizer, DecorrelateWithOtherPhiRegularizer2
)
from topnum.scores.intratext_coherence_score import (
    IntratextCoherenceScore,
    ComputationMethod,
    WordTopicRelatednessType,
)

In [4]:
import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

In [5]:
topicnet.__file__

! ls /home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager/

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [6]:
DATA_FOLDER_PATH = '/data_mil/shared/CompressaAI/iterative/data/noow'

In [7]:
! ls $DATA_FOLDER_PATH

_20_Newsgroups.csv	       Post_Science_NOOW.csv
_20_Newsgroups__internals      Post_Science_NOOW_fixed.csv
20_Newsgroups_NOOW.csv	       Post_Science_NOOW_fixed__internals
20_Newsgroups_NOOW__internals  Post_Science_NOOW__internals
_Lenta.csv		       WikiRef_220_NOOW.csv
MKB_10_NOOW.csv


In [8]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/MKB_10_NOOW.csv',
)

dataset.get_possible_modalities()

{'@letter', '@ngram', '@text'}

In [9]:
dataset._internals_folder_path

'/data_mil/shared/CompressaAI/iterative/data/noow/MKB_10_NOOW__internals'

In [10]:
MAIN_MODALITY = '@text'

In [11]:
dataset._data.head()

,id,raw_text,vw_text
id,,,
«Бедная_симптомами»_шизофрения,«Бедная_симптомами»_шизофрения,«Бе́дная симпто́мами» шизофрени́я — подтип шиз...,«Бедная_симптомами»_шизофрения |@text бедный с...
"46,XX/46,XY","46,XX/46,XY","46,XX/46,XY (тетрагаметный химеризм) — это раз...","46,XX/46,XY |@text <person> химеризм разновидн..."
"Синдром_48,_XXXY","Синдром_48,_XXXY","Синдром 48, XXXY — это генетическое состояние,...","Синдром_48,_XXXY |@text синдром xxxy генетичес..."
"Синдром_48,_XXYY","Синдром_48,_XXYY","Синдром 48, XXYY — это аномалия хромосом, при ...","Синдром_48,_XXYY |@text синдром xxyy аномалия ..."
"Синдром_48,_XYYY","Синдром_48,_XYYY","Синдром 48, XYYY — чрезвычайно редкая анеуплои...","Синдром_48,_XYYY |@text синдром xyyy чрезвычаи..."


In [12]:
dataset._data.shape

(2036, 3)

In [13]:
dataset.get_dictionary()

artm.Dictionary(name=bfc7d539-b553-4a85-bfc8-d10bfc0d71cf, num_entries=244551)

In [14]:
dictionary = dataset.get_dictionary()

In [15]:
print(dictionary)

for modality in dataset.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=bfc7d539-b553-4a85-bfc8-d10bfc0d71cf, num_entries=244551)


In [16]:
dictionary

artm.Dictionary(name=bfc7d539-b553-4a85-bfc8-d10bfc0d71cf, num_entries=46873)

In [17]:
dictionary.filter(min_df=2, max_df_rate=0.5)

artm.Dictionary(name=bfc7d539-b553-4a85-bfc8-d10bfc0d71cf, num_entries=22608)

In [17]:
dictionary.filter(min_df_rate=0.5)  # (min_df=2, max_df_rate=0.5)

artm.Dictionary(name=8e2e5cc7-556b-447e-a04d-7b1906a09bbf, num_entries=0)

In [34]:
dictionary

artm.Dictionary(name=3ed7db55-fc3e-4f86-b0ad-c4ffdcf82fd0, num_entries=12)

In [35]:
dictionary.save_text('test_dict.txt')

In [37]:
! cat test_dict.txt

name: 3ed7db55-fc3e-4f86-b0ad-c4ffdcf82fd0 num_items: 2036
token, class_id, token_value, token_tf, token_df
год, @text, 0.00550703052431345, 5426.0, 1094.0
лечение, @text, 0.005373059306293726, 5294.0, 1242.0
что, @text, 0.006797011010348797, 6697.0, 1244.0
как, @text, 0.0069807143881917, 6878.0, 1465.0
развитие, @text, 0.0036872541531920433, 3633.0, 1089.0
являться, @text, 0.005028996616601944, 4955.0, 1330.0
случай, @text, 0.005558792036026716, 5477.0, 1335.0
мочь, @text, 0.009810349904000759, 9666.0, 1602.0
<person>, @text, 0.04563852399587631, 44967.0, 1985.0
заболевание, @text, 0.006939101964235306, 6837.0, 1428.0
<person>_<person>, @ngram, 0.005488390568643808, 3171.0, 1053.0
быть, @text, 0.006246916949748993, 6155.0, 1330.0


In [18]:
dataset._cached_dict = dictionary

In [19]:
dataset.get_dictionary()

artm.Dictionary(name=bfc7d539-b553-4a85-bfc8-d10bfc0d71cf, num_entries=22608)

In [20]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [21]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 3.91 s, sys: 100 ms, total: 4.01 s
Wall time: 3.97 s


In [22]:
co_occurences.shape

(22608, 22608)

In [23]:
dataset.get_dictionary()

artm.Dictionary(name=bfc7d539-b553-4a85-bfc8-d10bfc0d71cf, num_entries=22608)

In [24]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [25]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [26]:
KnownModel

<enum 'KnownModel'>

In [27]:
PARAMS_EXPLORED

{<KnownModel.LDA: 'LDA'>: {'prior': ['symmetric', 'asymmetric', 'heuristic']},
 <KnownModel.PLSA: 'PLSA'>: {},
 <KnownModel.TLESS: 'TARTM'>: {},
 <KnownModel.SPARSE: 'sparse'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1]},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': [0.02,
   0.05,
   0.1]},
 <KnownModel.ARTM: 'ARTM'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1],
  'decorrelation_tau': [0.02, 0.05, 0.1]}}

In [28]:
NUM_TOPICS = 20  # vary
NUM_TRAINS = 3
NUM_ITERATIONS = 20
NUM_TOP_TOKENS = 20

In [29]:
dataset.get_dictionary()

artm.Dictionary(name=bfc7d539-b553-4a85-bfc8-d10bfc0d71cf, num_entries=22608)

In [30]:
dictionary = dataset.get_dictionary()

## Test

In [31]:
PARAMS_EXPLORED[KnownModel.PLSA]

{}

In [32]:
%%time

model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=NUM_TOPICS,
    seed=1,
)

model._fit(dataset.get_batch_vectorizer(), num_iterations=10)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


CPU times: user 19.8 s, sys: 231 ms, total: 20 s
Wall time: 10.3 s


In [33]:
list(model.scores.keys())

['PerplexityScore@all',
 'SparsityThetaScore',
 'SparsityPhiScore@text',
 'PerplexityScore@text',
 'TopicKernel@text.average_coherence',
 'TopicKernel@text.average_contrast',
 'TopicKernel@text.average_purity',
 'TopicKernel@text.average_size',
 'TopicKernel@text.coherence',
 'TopicKernel@text.contrast',
 'TopicKernel@text.purity',
 'TopicKernel@text.size',
 'TopicKernel@text.tokens']

In [34]:
model.scores[f'PerplexityScore{MAIN_MODALITY}']

[22190.142578125,
 4108.90380859375,
 3652.6328125,
 3045.193603515625,
 2687.217529296875,
 2507.751953125,
 2406.8203125,
 2344.760009765625,
 2304.6728515625,
 2277.6640625]

In [59]:
phi = model.get_phi()
target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
target_topic_names = [phi.columns[i] for i in target_topic_indices]

custom_scores = [
    TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )
    for top in [20]  # [10, 20, 50, 100]
]
custom_scores = custom_scores + [
    DiversityScore(
        name=f'diversity_{metric}',
        topic_names=['topic_0', 'topic_1'],
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

for score in custom_scores:
    res = score.call(model)

    print(score._name)
    print(res)

    if isinstance(score, TopTokenCoherence):
        res_by_topic = score.call_by_topic(model)

        print(res_by_topic)

coherence_20
[0.0273816]
{0: array([0.]), 1: array([0.02958778]), 2: array([0.04763114]), 3: array([0.01888187]), 4: array([0.]), 5: array([0.03611333]), 6: array([0.]), 7: array([0.09164026]), 8: array([0.04763114]), 9: array([0.]), 10: array([0.]), 11: array([0.07011646]), 12: array([0.]), 13: array([0.]), 14: array([0.]), 15: array([0.]), 16: array([0.]), 17: array([0.08207206]), 18: array([0.02869565]), 19: array([0.09526229])}
diversity_euclidean
0.038674582514403096
diversity_jensenshannon
0.038674582514403096
diversity_hellinger
0.038674582514403096
diversity_cosine
0.038674582514403096


In [35]:
intra = IntratextCoherenceScore(
    name='toplen',
    data=dataset,
    computation_method=ComputationMethod.SEGMENT_LENGTH,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
)

In [36]:
%%time

intra.call(model)

CPU times: user 1min 3s, sys: 1.14 s, total: 1min 4s
Wall time: 1min 1s


2.028455951901683

In [37]:
model.get_phi(class_ids=MAIN_MODALITY)['topic_18'].sort_values(ascending=False)

modality  token            
@text     половый              0.022724
          мужчина              0.009491
          женщина              0.009099
          яичко                0.005944
          время                0.005493
                                 ...   
          побережье            0.000000
          радужка              0.000000
          интервенционноить    0.000000
          шизоаффективный      0.000000
          хелирование          0.000000
Name: topic_18, Length: 22608, dtype: float32

In [38]:
model.class_ids

{'@text': 1}

In [39]:
KNOWN_METRICS

['euclidean', 'jensenshannon', 'hellinger', 'cosine']

In [40]:
MAIN_MODALITY

'@text'

In [41]:
def fit_and_compute_scores(model, dataset):
    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    
    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    # print(f'Computing "{coherence_score._name}"...')
    
    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    # print(f'Result by topic: {topic_coherences}.')


    # intra1 = IntratextCoherenceScore(
    #     name='toplen_pwt',
    #     data=dataset,
    #     computation_method=ComputationMethod.SEGMENT_LENGTH,
    #     word_topic_relatedness=WordTopicRelatednessType.PWT,
    #     should_compute=False,  # only on last iter
    # )
    intra2 = IntratextCoherenceScore(
        name='toplen_ptw',
        data=dataset,
        computation_method=ComputationMethod.SEGMENT_LENGTH,
        word_topic_relatedness=WordTopicRelatednessType.PTW,
        should_compute=False,
    )
    # intra3 = IntratextCoherenceScore(
    #     name='topden_ptw',
    #     data=dataset,
    #     computation_method=ComputationMethod.SUM_OVER_WINDOW,
    #     word_topic_relatedness=WordTopicRelatednessType.PTW,
    #     should_compute=False,
    # )
    # intra3_w4 = IntratextCoherenceScore(
    #     name='topden_ptw',
    #     data=dataset,
    #     computation_method=ComputationMethod.SUM_OVER_WINDOW,
    #     word_topic_relatedness=WordTopicRelatednessType.PTW,
    #     window=4,
    #     should_compute=False,
    # )

    intra_topic_coherences = dict()

    for intra in [intra2]:  # [intra2, intra3_w4]:  #[intra1, intra2, intra3]:
        # print(f'\nComputing "{intra._name}"...')

        current_intra_topic_coherences = intra.compute(model)

        assert all(v is not None for v in current_intra_topic_coherences.values())

        _values = current_intra_topic_coherences.values()

        current_intra_topic_coherences = {
            i: current_intra_topic_coherences[t]  # if v is not None else 0.0
            for i, t in enumerate(target_topic_names)
        }

        assert all(abs(x - y) <= 1e-6 for x, y in zip(_values, current_intra_topic_coherences.values())), (_values, current_intra_topic_coherences.values())  # "sorted" Python dicts
        
        intra_topic_coherences[f'topic_coherences_{intra._name}'] = current_intra_topic_coherences

        value = float(np.median(list(current_intra_topic_coherences.values())))
        score_values[intra._name] = value

        # print(f'Result by topic: {current_intra_topic_coherences}.')


    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    
    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
        **intra_topic_coherences,
    }

In [42]:
co_occurences.shape

(22608, 22608)

In [58]:
BEST_PARAMS = dict()

## PLSA

In [59]:
PARAMS_EXPLORED[KnownModel.PLSA]

{}

In [60]:
NUM_TOPICS

20

In [61]:
results = []

for seed in range(NUM_TRAINS):
    print(seed)
    
    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    scores = fit_and_compute_scores(model, dataset)
    results.append(scores)

0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [62]:
BEST_PARAMS[KnownModel.PLSA] = None

In [63]:
dataset._data.shape

(2036, 3)

## Sparse

In [64]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [65]:
results = dict()

for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['sparse_sp_tau']:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (sparse_sp_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.SPARSE,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={'sparse_sp_tau': sparse_sp_tau, 'smooth_bcg_tau': smooth_bcg_tau}
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
 
            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(-0.05, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.08920314764263809
sparse_theta_sp: -0.9905229675367199
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.08920314764263809
sparse_theta_sp: -0.9905229675367199
2


...

smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: -0.17029691822685453
sparse_theta_sp: -1.8909983925701015
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: -0.17029691822685453
sparse_theta_sp: -1.8909983925701015
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: -0.17029691822685453
sparse_theta_sp: -1.8909983925701015




In [66]:
results

{(-0.05,
  0.05): [{'scores': {'perplexity': 2465.832275390625,
    'coherence_20': array([0.65723241]),
    'diversity_euclidean': 0.06269599207736247,
    'diversity_jensenshannon': 0.6402356058322131,
    'diversity_hellinger': 0.7457255319298899,
    'diversity_cosine': 0.7392224332902931},
   'topic_coherences': {0: 0.4706941705736568,
    1: 0.9428906252054198,
    2: 0.570444932146175,
    3: 0.730668002345812,
    4: 0.5060028501686357,
    5: 0.4089519532813123,
    6: 0.91137477735875,
    7: 0.618962659949188,
    8: 0.5312373596697549,
    9: 0.6546315002705506,
    10: 0.8032429711947393,
    11: 0.5274397261902024,
    12: 0.8307535413071001,
    13: 0.6573898814990363,
    14: 0.46222355277065214,
    15: 0.6544428741574582,
    16: 0.9044616779054204,
    17: 0.6078315922424556,
    18: 0.6808253694723042,
    19: 0.6701780840189774}}, {'scores': {'perplexity': 2465.54345703125,
    'coherence_20': array([0.62574678]),
    'diversity_euclidean': 0.06272248450068274,
   

In [68]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

(-0.05, 0.05) 2454.600341796875
(-0.05, 0.1) 2577.962646484375
(-0.1, 0.05) 2569.99609375
(-0.1, 0.1) 2695.2994791666665


In [69]:
# Best: (-0.05, 0.05) 2454.600341796875

In [70]:
BEST_PARAMS[KnownModel.SPARSE] = {
    'sparse_sp_tau': -0.05,
    'smooth_bcg_tau': 0.05,
}

In [71]:
model.get_phi()['topic_15'].sort_values(ascending=False)

modality  token       
@text     расстроиство    0.041009
          личность        0.013708
          шизофрения      0.012495
          человек         0.011630
          психический     0.008991
                            ...   
          наслаиваться    0.000000
          скелетный       0.000000
          валик           0.000000
          угольный        0.000000
          легально        0.000000
Name: topic_15, Length: 22608, dtype: float32

## Decorrelation

In [72]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [73]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [74]:
DECORRELATION_TAUS = [0.01] + PARAMS_EXPLORED[KnownModel.DECORRELATION]['decorrelation_tau']

In [75]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (decorrelation_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': decorrelation_tau,
                    'smooth_bcg_tau': smooth_bcg_tau,
                    'sparse_sp_tau': 0.0,
                }
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")

            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(0.01, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
2


...

smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1




In [79]:
len(results)

8

In [80]:
results

{(0.01,
  0.05): [{'scores': {'perplexity': 2269.100830078125,
    'coherence_20': array([0.6447714]),
    'diversity_euclidean': 0.05453546771942422,
    'diversity_jensenshannon': 0.6065282447749323,
    'diversity_hellinger': 0.6995732124401457,
    'diversity_cosine': 0.7049150689406795},
   'topic_coherences': {0: 0.4938616070784748,
    1: 0.8371804775325153,
    2: 0.4438504796964468,
    3: 0.7285835210890776,
    4: 0.5078567786356974,
    5: 0.4541532351151352,
    6: 0.8310553439006045,
    7: 0.6032429603377425,
    8: 0.808188785425736,
    9: 0.5908592555975367,
    10: 0.8160658668511684,
    11: 0.5530545787668294,
    12: 0.8281908937052933,
    13: 0.5575985763907658,
    14: 0.4901710044759395,
    15: 0.6330720539594581,
    16: 0.9980639632604971,
    17: 0.6184429737039088,
    18: 0.6046728654308625,
    19: 0.4972627448804702}}, {'scores': {'perplexity': 2263.556396484375,
    'coherence_20': array([0.64126859]),
    'diversity_euclidean': 0.056831659778020335,


In [81]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)

    print(k, mean_ppl)

(0.01, 0.05) 2257.5345052083335
(0.01, 0.1) 2361.4169921875
(0.02, 0.05) 2257.9066569010415
(0.02, 0.1) 2361.5891927083335
(0.05, 0.05) 2269.4532063802085
(0.05, 0.1) 2372.4910481770835
(0.1, 0.05) 2345.3286946614585
(0.1, 0.1) 2440.0069986979165


In [82]:
#  Best:               (0.01, 0.05) 2257.5345052083335
# Close (very close): (0.02, 0.05) 2257.9066569010415

In [83]:
BEST_PARAMS[KnownModel.DECORRELATION] = {
    'decorrelation_tau': 0.02,
    'smooth_bcg_tau': 0.05,
}

## ARTM

In [84]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [85]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [86]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [87]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.ARTM]['sparse_sp_tau']:
        for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
            key = (decorrelation_tau, sparse_sp_tau, smooth_bcg_tau)
            results[key] = []
    
            print(key)
    
            for seed in range(NUM_TRAINS):
                print(seed)
                
                model = init_model_from_family(
                    family=KnownModel.ARTM,
                    dataset=dataset,
                    main_modality=MAIN_MODALITY,
                    num_topics=NUM_TOPICS,
                    seed=seed,
                    model_params={
                        'decorrelation_tau': decorrelation_tau,
                        'smooth_bcg_tau': smooth_bcg_tau,
                        'sparse_sp_tau': sparse_sp_tau,
                    }
                )
    
                for reg in model.regularizers.data:
                    print(f"{reg}: {model.regularizers[reg].tau}")
    
                scores = fit_and_compute_scores(model, dataset)
                results[key].append(scores)

            print()

        print()

    print()

(0.01, -0.05, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.08920314764263809
sparse_theta_sp: -0.9905229675367199
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.08920314764263809
sparse_theta_sp: -0.9905229675367199
decorrelation: 0.01
2


...

smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: -0.17029691822685453
sparse_theta_sp: -1.8909983925701015
decorrelation: 0.1
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: -0.17029691822685453
sparse_theta_sp: -1.8909983925701015
decorrelation: 0.1
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: -0.17029691822685453
sparse_theta_sp: -1.8909983925701015
decorrelation: 0.1





In [91]:
len(results)

16

In [92]:
ppls = []

for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    ppls.append(mean_ppl)

    print(k, mean_ppl)

(0.01, -0.05, 0.05) 2458.5643717447915
(0.01, -0.05, 0.1) 2581.1787109375
(0.01, -0.1, 0.05) 2574.4801432291665
(0.01, -0.1, 0.1) 2698.272705078125
(0.02, -0.05, 0.05) 2464.08544921875
(0.02, -0.05, 0.1) 2585.5170084635415
(0.02, -0.1, 0.05) 2579.8678385416665
(0.02, -0.1, 0.1) 2702.2215983072915
(0.05, -0.05, 0.05) 2488.5302734375
(0.05, -0.05, 0.1) 2603.748291015625
(0.05, -0.1, 0.05) 2600.926025390625
(0.05, -0.1, 0.1) 2718.6932779947915
(0.1, -0.05, 0.05) 2552.7762858072915
(0.1, -0.05, 0.1) 2658.8765462239585
(0.1, -0.1, 0.05) 2662.7177734375
(0.1, -0.1, 0.1) 2775.323974609375


In [93]:
sorted(ppls)[:5]

[2458.5643717447915,
 2464.08544921875,
 2488.5302734375,
 2552.7762858072915,
 2574.4801432291665]

In [ ]:
#  Best: (0.01, -0.05, 0.05) 2458.5643717447915
# Close: (0.02, -0.05, 0.05) 2464.08544921875

In [94]:
BEST_PARAMS[KnownModel.DECORRELATION] = {
    'decorrelation_tau': 0.01,
    'sparse_sp_tau':    -0.05,
    'smooth_bcg_tau':    0.05,
}

## TLESS

In [95]:
PARAMS_EXPLORED[KnownModel.TLESS]

{}

In [96]:
results = []

for seed in range(NUM_TRAINS):
    print(seed)
    
    model = init_model_from_family(
        family=KnownModel.TLESS,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    scores = fit_and_compute_scores(model, dataset)
    results.append(scores)

0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [97]:
results

[{'scores': {'perplexity': 2710.05322265625,
   'coherence_20': array([0.69064164]),
   'diversity_euclidean': 0.10360478401886986,
   'diversity_jensenshannon': 0.7566075022840391,
   'diversity_hellinger': 0.8854519637041928,
   'diversity_cosine': 0.9298865715901761},
  'topic_coherences': {0: 0.4369066763829261,
   1: 1.3688887093860056,
   2: 0.5734527878692386,
   3: 0.555006268470112,
   4: 0.5031285424754262,
   5: 0.5165577102590853,
   6: 0.917272974090113,
   7: 0.8170612891678282,
   8: 0.6308371342605484,
   9: 0.6249737112288283,
   10: 0.7587532979791912,
   11: 0.9109564791512955,
   12: 0.7263835755690161,
   13: 0.6870362602850006,
   14: 0.35765613593653406,
   15: 0.6732874851412556,
   16: 0.8520000816925969,
   17: 0.4636096696813935,
   18: 1.0211192239453564,
   19: 0.41794472443348246}},
 {'scores': {'perplexity': 2713.791259765625,
   'coherence_20': array([0.66324343]),
   'diversity_euclidean': 0.10462702946308376,
   'diversity_jensenshannon': 0.75628969734

In [98]:
# Best:

In [99]:
BEST_PARAMS[KnownModel.TLESS] = None

## LDA

In [100]:
PARAMS_EXPLORED[KnownModel.LDA]

{'prior': ['symmetric', 'asymmetric', 'heuristic']}

In [101]:
results = dict()

for prior in PARAMS_EXPLORED[KnownModel.LDA]['prior']:
    key = prior
    results[key] = []

    print(key)

    for seed in range(NUM_TRAINS):
        print(seed)
        
        model = init_model_from_family(
            family=KnownModel.LDA,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params={'prior': prior}
        )

        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

        scores = fit_and_compute_scores(model, dataset)
        results[key].append(scores)

    print()

symmetric
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
2


...

smooth_phi: 0.01
smooth_theta: 2.5
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5



In [105]:
results

{'symmetric': [{'scores': {'perplexity': 2206.36328125,
    'coherence_20': array([0.58326465]),
    'diversity_euclidean': 0.04829135840773115,
    'diversity_jensenshannon': 0.5641920329897929,
    'diversity_hellinger': 0.6244635499992061,
    'diversity_cosine': 0.6530752667822146},
   'topic_coherences': {0: 0.41603726501551547,
    1: 0.7613769777112703,
    2: 0.4178029383666581,
    3: 0.649953932720543,
    4: 0.4980276984955354,
    5: 0.48879998830907045,
    6: 0.7867876162181322,
    7: 0.49511778383633126,
    8: 0.705775085791701,
    9: 0.5609756518683418,
    10: 0.7079941341971158,
    11: 0.5274557253976083,
    12: 0.7109965752522808,
    13: 0.5510059357376327,
    14: 0.3947668738392959,
    15: 0.46749608493419537,
    16: 0.7961542891063786,
    17: 0.6018775126751085,
    18: 0.6035551172078581,
    19: 0.5233358795514585}},
  {'scores': {'perplexity': 2201.345703125,
    'coherence_20': array([0.58114333]),
    'diversity_euclidean': 0.05324276936292553,
    '

In [106]:
ppls = []

for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    ppls.append(mean_ppl)

    print(k, mean_ppl)

symmetric 2195.3936360677085
asymmetric 2195.6841634114585
heuristic 2337.3949381510415


In [104]:
sorted(ppls)

[2195.3936360677085, 2195.6841634114585, 2337.3949381510415]

In [94]:
# Best:               asymmetric 2195.6841634114585
# Close (very close): symmetric 2195.3936360677085

In [107]:
BEST_PARAMS[KnownModel.LDA] = {
    'prior': 'symmetric',
}

In [108]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'symmetric'}}

In [43]:
BEST_PARAMS = {KnownModel.PLSA: None,
 KnownModel.SPARSE: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 KnownModel.DECORRELATION: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 KnownModel.TLESS: None,
 KnownModel.LDA: {'prior': 'symmetric'}}

In [44]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'symmetric'}}

In [45]:
import json
import warnings

warnings.simplefilter('ignore', UserWarning)

In [46]:
NUM_TRAINS = 20  # 100
COHERENCES = {
    'topic_coherences': list(),
    # 'topic_coherences_toplen_pwt': list(),
    'topic_coherences_toplen_ptw': list(),
    # 'topic_coherences_topden_ptw': list(),
}

In [44]:
! ls results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [47]:
SAVE_FOLDER = 'results_intra/mkb10'

! mkdir -p $SAVE_FOLDER

In [48]:
! ls results_intra

20newsgroups  mkb10  postnauka


In [49]:
# PLSA

start = time.time()

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    # COHERENCES.extend(
    #     list(results['topic_coherences'].values())
    # )

    for k in COHERENCES:
        COHERENCES[k].extend(
            list(results[k].values())
        )

end = time.time()

print(f'Elapsed: {(end - start) / 60} min')

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 28.559891831874847 min


In [52]:
scores[0]

{'scores': {'perplexity': 2183.57958984375,
  'coherence_20': array([0.58371909]),
  'diversity_euclidean': 0.049940201816554355,
  'diversity_jensenshannon': 0.5937372193914952,
  'diversity_hellinger': 0.6835495470339691,
  'diversity_cosine': 0.6570792108339262},
 'topic_coherences': {0: 0.42962387858537326,
  1: 0.7701415600860497,
  2: 0.40759208792425244,
  3: 0.649953932720543,
  4: 0.49802769849553535,
  5: 0.4887999883090705,
  6: 0.8222052116329212,
  7: 0.5338939890004764,
  8: 0.7126840568737308,
  9: 0.5541666824798857,
  10: 0.6344829013287363,
  11: 0.4364507624526562,
  12: 0.7874127543914287,
  13: 0.6213442459066951,
  14: 0.37798571771838246,
  15: 0.48500124462101596,
  16: 0.7940350614488233,
  17: 0.6018775126751085,
  18: 0.5565705317622954,
  19: 0.5121320595317724}}

In [50]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [117]:
scores

[{'perplexity': 2183.57958984375,
  'coherence_20': 0.5837190938972376,
  'diversity_euclidean': 0.0499402027128018,
  'diversity_jensenshannon': 0.5937372250477058,
  'diversity_hellinger': 0.6835495478309515,
  'diversity_cosine': 0.6570792281197485},
 {'perplexity': 2182.065185546875,
  'coherence_20': 0.585073631542489,
  'diversity_euclidean': 0.054420685001269314,
  'diversity_jensenshannon': 0.5973725609648748,
  'diversity_hellinger': 0.6877445355440452,
  'diversity_cosine': 0.696404372046248},
 {'perplexity': 2160.20263671875,
  'coherence_20': 0.5903965196666497,
  'diversity_euclidean': 0.05219890614430589,
  'diversity_jensenshannon': 0.5977495295059344,
  'diversity_hellinger': 0.6879514712514758,
  'diversity_cosine': 0.6830169600074231},
 {'perplexity': 2158.900390625,
  'coherence_20': 0.6188199121004773,
  'diversity_euclidean': 0.05387041411544235,
  'diversity_jensenshannon': 0.6057371032917634,
  'diversity_hellinger': 0.6978361317969153,
  'diversity_cosine': 0.70

In [51]:
SAVE_FOLDER

'results_intra/mkb10'

In [52]:
with open(SAVE_FOLDER + '/plsa_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [53]:
COHERENCES

{'topic_coherences': [0.42962387858537326,
  0.7701415600860497,
  0.40759208792425244,
  0.649953932720543,
  0.49802769849553535,
  0.4887999883090705,
  0.8222052116329212,
  0.5338939890004764,
  0.7126840568737308,
  0.5541666824798857,
  0.6344829013287363,
  0.4364507624526562,
  0.7874127543914287,
  0.6213442459066951,
  0.37798571771838246,
  0.48500124462101596,
  0.7940350614488233,
  0.6018775126751085,
  0.5565705317622954,
  0.5121320595317724,
  0.7627239882615724,
  0.6071893841189011,
  0.3411872834044675,
  0.4467820828340276,
  0.5133054280900867,
  0.5884674679701702,
  0.7566166592486294,
  0.4734190345238171,
  0.5709806439030034,
  0.6958666641224664,
  0.5061710203348446,
  0.7462137316823558,
  0.8343641651596831,
  0.5651634303590395,
  0.6783977585819944,
  0.4842819292557047,
  0.49264268632429903,
  0.45233410584274264,
  0.7222533186553418,
  0.4631118481766318,
  0.5273192179072478,
  0.4560709941717536,
  0.487859585128513,
  0.7671496414768496,
  0.386

In [120]:
len(COHERENCES)

400

In [121]:
COHERENCES[:10]

[0.42962387858537326,
 0.7701415600860497,
 0.40759208792425244,
 0.649953932720543,
 0.49802769849553535,
 0.4887999883090705,
 0.8222052116329212,
 0.5338939890004764,
 0.7126840568737308,
 0.5541666824798857]

In [122]:
COHERENCES[:20]

[0.42962387858537326,
 0.7701415600860497,
 0.40759208792425244,
 0.649953932720543,
 0.49802769849553535,
 0.4887999883090705,
 0.8222052116329212,
 0.5338939890004764,
 0.7126840568737308,
 0.5541666824798857,
 0.6344829013287363,
 0.4364507624526562,
 0.7874127543914287,
 0.6213442459066951,
 0.37798571771838246,
 0.48500124462101596,
 0.7940350614488233,
 0.6018775126751085,
 0.5565705317622954,
 0.5121320595317724]

In [54]:
# Sparse

start = time.time()

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.SPARSE,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
        model_params=BEST_PARAMS[KnownModel.SPARSE],
    )

    if seed == 0:
        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    # COHERENCES.extend(
    #     list(results['topic_coherences'].values())
    # )

    for k in COHERENCES:
        COHERENCES[k].extend(
            list(results[k].values())
        )

end = time.time()

print(f'Elapsed: {(end - start) / 60} min')

0 smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.08920314764263809
sparse_theta_sp: -0.9905229675367199
1 2 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 31.686683456103008 min


In [55]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [56]:
with open(SAVE_FOLDER + '/sparse_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [126]:
len(COHERENCES)

800

In [127]:
COHERENCES[-10:]

[0.593084505505161,
 0.5139246158216321,
 0.4536524649784854,
 0.8882307492852554,
 0.7211801093949745,
 0.6918342537655878,
 0.5452751336539511,
 0.8003961452414058,
 0.4244178592187105,
 0.5774470345256705]

In [128]:
max(COHERENCES)

1.1430544730916514

In [129]:
min(COHERENCES)

0.3008890091768996

In [57]:
def train_many(model_family, save_file_path):
    start = time.time()
    
    scores = []

    # for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
    for seed in range(NUM_TRAINS):
        if seed != NUM_TRAINS - 1:
            print(seed, end=' ')
        else:
            print(seed)
    
        model = init_model_from_family(
            family=model_family,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params=BEST_PARAMS[model_family],
        )
    
        if seed == 0:
            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
    
        results = fit_and_compute_scores(model, dataset)
        # scores.append(results['scores'])
        scores.append(results)
    
        # COHERENCES.extend(
        #     list(results['topic_coherences'].values())
        # )
    
        for k in COHERENCES:
            COHERENCES[k].extend(
                list(results[k].values())
            )
    
    end = time.time()
    
    print(f'Elapsed: {(end - start) / 60} min')

    # for s in scores:
    #     s['coherence_20'] = float(s['coherence_20'])
    
    for s in scores:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

    with open(save_file_path, 'w') as f:
        f.write(
            json.dumps(scores, indent=4)
        )

In [58]:
train_many(KnownModel.DECORRELATION, SAVE_FOLDER + '/decorrelation_with_cohs.json')

0 decorrelation: 0.01
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 30.349491894245148 min


In [132]:
len(COHERENCES)

1200

In [59]:
train_many(KnownModel.TLESS, SAVE_FOLDER + '/tless_with_cohs.json')

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 32.29338770707448 min


In [134]:
len(COHERENCES)

1600

In [60]:
train_many(KnownModel.LDA, SAVE_FOLDER + '/lda_with_cohs.json')

0 smooth_phi: 0.05
smooth_theta: 0.05
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 30.342467697461448 min


In [63]:
len(COHERENCES)

2000

In [137]:
COHERENCES

[0.42962387858537326,
 0.7701415600860497,
 0.40759208792425244,
 0.649953932720543,
 0.49802769849553535,
 0.4887999883090705,
 0.8222052116329212,
 0.5338939890004764,
 0.7126840568737308,
 0.5541666824798857,
 0.6344829013287363,
 0.4364507624526562,
 0.7874127543914287,
 0.6213442459066951,
 0.37798571771838246,
 0.48500124462101596,
 0.7940350614488233,
 0.6018775126751085,
 0.5565705317622954,
 0.5121320595317724,
 0.7627239882615724,
 0.6071893841189011,
 0.3411872834044675,
 0.4467820828340276,
 0.5133054280900867,
 0.5884674679701702,
 0.7566166592486294,
 0.4734190345238171,
 0.5709806439030034,
 0.6958666641224664,
 0.5061710203348446,
 0.7462137316823558,
 0.8343641651596831,
 0.5651634303590395,
 0.6783977585819944,
 0.4842819292557047,
 0.49264268632429903,
 0.45233410584274264,
 0.7222533186553418,
 0.4631118481766318,
 0.5273192179072478,
 0.4560709941717536,
 0.487859585128513,
 0.7671496414768496,
 0.3863527192969232,
 0.49826597310962606,
 0.6392971544587822,
 0.5877

In [61]:
COHERENCES.keys()

dict_keys(['topic_coherences', 'topic_coherences_toplen_ptw'])

In [62]:
for k in COHERENCES:
    print(k)
    print(len(COHERENCES[k]))

topic_coherences
2000
topic_coherences_toplen_ptw
2000


In [81]:
! ls results_intra/mkb10

decorrelation_with_cohs.json  plsa_with_cohs.json    tless_with_cohs.json
lda_with_cohs.json	      sparse_with_cohs.json


In [83]:
for k in COHERENCES:
    print(k)

    for p in range(5, 100, 5):
        print(f'{p:2}: {np.percentile(COHERENCES[k], p)}')

    print()

topic_coherences
 5: 0.4023442717915072
10: 0.4407050725684525
15: 0.468809417987108
20: 0.48898656658973
25: 0.5080421609016634
30: 0.5268174078450744
35: 0.5452060481399934
40: 0.5645901643197999
45: 0.5791220119147815
50: 0.5964218853337424
55: 0.6154604950971919
60: 0.6321741710189025
65: 0.652730127881449
70: 0.6732219845449956
75: 0.7031854695257616
80: 0.7372573006249383
85: 0.7677269697946429
90: 0.8110348193665089
95: 0.9002260505382171

topic_coherences_toplen_ptw
 5: 1.6560796452007274
10: 1.7434408568528794
15: 1.8178484344823411
20: 1.8676456834336237
25: 1.9050773720690417
30: 1.9489293661997766
35: 1.9944735555627704
40: 2.034768845954679
45: 2.076784939420604
50: 2.1205828270498492
55: 2.1696782813786726
60: 2.220653524692646
65: 2.273674951130135
70: 2.344754594835432
75: 2.4166349619746246
80: 2.5358203425535235
85: 2.6793444091996115
90: 2.8621394146113888
95: 3.224302090987209



In [139]:
min(COHERENCES), max(COHERENCES)

(0.23452606480903626, 1.3923473516174445)

In [80]:
for k in COHERENCES:
    print(k)
    print(min(COHERENCES[k]), max(COHERENCES[k]))
    print()

topic_coherences
0.23452606480903626 1.3923473516174445

topic_coherences_toplen_ptw
1.4621913836963019 4.295964905348685



In [145]:
np.argmin(COHERENCES), np.argmax(COHERENCES)

(879, 1385)

In [146]:
for p in [2, 98]:
    print(f'{p:2}: {np.percentile(COHERENCES, p)}')

 2: 0.3576196853316383
98: 1.0211518252823666


In [142]:
1

1

In [84]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'symmetric'}}

In [85]:
dataset.get_dictionary()

artm.Dictionary(name=bfc7d539-b553-4a85-bfc8-d10bfc0d71cf, num_entries=22608)

In [138]:
COHERENCES[-11:]

[0.9151910589643388,
 0.40117780835167877,
 0.4405340306270155,
 0.4395601942762919,
 0.58209845522529,
 0.6616468035210405,
 0.4973913152802322,
 0.5207985518820669,
 0.6030229525188386,
 1.2961590770289348,
 0.6465084701742573]

In [86]:
COHERENCES

{'topic_coherences': [0.42962387858537326,
  0.7701415600860497,
  0.40759208792425244,
  0.649953932720543,
  0.49802769849553535,
  0.4887999883090705,
  0.8222052116329212,
  0.5338939890004764,
  0.7126840568737308,
  0.5541666824798857,
  0.6344829013287363,
  0.4364507624526562,
  0.7874127543914287,
  0.6213442459066951,
  0.37798571771838246,
  0.48500124462101596,
  0.7940350614488233,
  0.6018775126751085,
  0.5565705317622954,
  0.5121320595317724,
  0.7627239882615724,
  0.6071893841189011,
  0.3411872834044675,
  0.4467820828340276,
  0.5133054280900867,
  0.5884674679701702,
  0.7566166592486294,
  0.4734190345238171,
  0.5709806439030034,
  0.6958666641224664,
  0.5061710203348446,
  0.7462137316823558,
  0.8343641651596831,
  0.5651634303590395,
  0.6783977585819944,
  0.4842819292557047,
  0.49264268632429903,
  0.45233410584274264,
  0.7222533186553418,
  0.4631118481766318,
  0.5273192179072478,
  0.4560709941717536,
  0.487859585128513,
  0.7671496414768496,
  0.386

## Newman Vs. Intratext

### Correlation

In [66]:
COHERENCES.keys()

dict_keys(['topic_coherences', 'topic_coherences_toplen_ptw'])

In [67]:
from scipy.stats import spearmanr

In [68]:
spearmanr(
    COHERENCES['topic_coherences'],
    COHERENCES['topic_coherences_toplen_ptw'],
)

SignificanceResult(statistic=0.17048267013018176, pvalue=1.6451101452195505e-14)

### Tops Intersection

In [69]:
inds1 = np.argsort(COHERENCES['topic_coherences'])[::-1][:100]

In [70]:
inds1 = set(np.argsort(COHERENCES['topic_coherences'])[::-1][:100])
inds2 = set(np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:100])

In [71]:
len(inds1 & inds2)

22

In [72]:
inds1 = set(np.argsort(COHERENCES['topic_coherences'])[::-1][:500])
inds2 = set(np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:500])

In [73]:
len(inds1 & inds2)

179

### Newman-in-Intratext Density (Mutual Density / Intra-Top Density)

In [74]:
inds2 = np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:100]

In [75]:
np.mean(
    np.array(COHERENCES['topic_coherences'])
)

0.6167487510943762

In [76]:
np.mean(
    np.array(COHERENCES['topic_coherences'])[inds2]
)

0.7445550893459846

In [77]:
inds2 = np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:200]

In [78]:
np.mean(
    np.array(COHERENCES['topic_coherences'])[inds2]
)

0.7039172695883679